# DriverlessCars — Google Colab (GPU)

1. **Runtime → Change runtime type → Hardware accelerator → GPU** (then Save).
2. **Run all cells in order** (Runtime → Run all) — do not skip the clone cell.
3. The last cell prints a **gradio.live** public URL — open it to use the demo.

**Reopened the notebook but UI looks old?** Closing/reopening does **not** update `/content/DriverlessCars`. After any GitHub push, either:
- **Runtime → Restart session**, then run **all** cells from the top, **or**
- Re-run only the **clone** cell (deletes old code) and then install + launch.

**Fresh notebook from GitHub:** [Open in Colab](https://colab.research.google.com/github/ShahramChaudhry/DriverlessCars/blob/claude/autonomous-driving-demo-Ljmeg/colab/DriverlessCars_Colab.ipynb) (avoids an old copy saved in your Google Drive).

**Get the code on Colab** (pick one):
- **Public GitHub:** run the clone cell below. If you forked the repo, edit `REPO_URL` there.
- **If `git clone` fails with exit 128:** confirm the repo is **Public** on GitHub.
- **If stderr says `Unable to read current working directory`:** **Runtime → Restart session**, run the first code cell (`os.chdir("/content")`), then clone again.
- **Private / no GitHub:** upload a zip to `/content/DriverlessCars` and skip clone.

In [ ]:
import os

os.chdir("/content")

!nvidia-smi
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

### Clone / refresh code from GitHub
**Run this every time** you want the latest UI and fixes (it replaces `/content/DriverlessCars`).

In [ ]:
%cd /content

REPO_URL = "https://github.com/ShahramChaudhry/DriverlessCars.git"
BRANCH = "claude/autonomous-driving-demo-Ljmeg"

!rm -rf /content/DriverlessCars
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/DriverlessCars

%cd /content/DriverlessCars
!git log -1 --oneline
print("Clone OK — commit above should match latest on GitHub")

### Install dependencies
Colab usually ships with CUDA-enabled PyTorch; `requirements.txt` may upgrade it to match Ultralytics.

In [ ]:
%cd /content/DriverlessCars
!pip install -q -r requirements.txt

### Launch Gradio (public link)
Wait until you see a **gradio.app** / **trycloudflare.com** URL, then click it. Keep this cell running.

In [ ]:
%cd /content/DriverlessCars
import importlib
import subprocess
import sys

APP = "/content/DriverlessCars/app.py"
REPO = "/content/DriverlessCars"

r = subprocess.run(
    ["git", "-C", REPO, "log", "-1", "--oneline"],
    capture_output=True,
    text=True,
)
if r.returncode == 0:
    print("Git:", r.stdout.strip())
    # Quick pull if you re-ran launch without re-cloning (optional refresh)
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
    r2 = subprocess.run(
        ["git", "-C", REPO, "log", "-1", "--oneline"],
        capture_output=True,
        text=True,
    )
    if r2.returncode == 0:
        print("After pull:", r2.stdout.strip())
else:
    print("No git repo — re-run the clone cell above.")

import py_compile

py_compile.compile(APP, doraise=True)
print("app.py syntax OK")

if REPO not in sys.path:
    sys.path.insert(0, REPO)

# Bust Colab's cached import if you re-ran this cell without restarting
if "app" in sys.modules:
    import app

    importlib.reload(app)
    launch_gradio = app.launch_gradio
else:
    from app import launch_gradio

launch_gradio(share=True)